In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QUICK-PDE: Eine Qiskit Function von ColibriTD
*Siehe die [API-Referenz](https://docs.quantum.ibm.com/api/functions/colibritd-pde)*

> **Note:** Qiskit Functions sind eine experimentelle Funktion, die Benutzern des IBM Quantum&reg; Premium Plans, Flex Plans und On-Prem (über IBM Quantum Platform API) Plans zur Verfügung steht. Sie befinden sich im Preview-Release-Status und können sich ändern.
## Überblick
Der hier vorgestellte Solver für partielle Differentialgleichungen (PDE) ist Teil unserer Quantum Innovative Computing Kit (QUICK)-Plattform (QUICK-PDE) und wird als Qiskit Function bereitgestellt. Mit der QUICK-PDE-Funktion kannst du domänenspezifische partielle Differentialgleichungen auf IBM Quantum QPUs lösen. Diese Funktion basiert auf dem in [ColibriTDs H-DES-Beschreibungspapier](https://arxiv.org/abs/2410.01130) beschriebenen Algorithmus. Dieser Algorithmus kann komplexe Multiphysik-Probleme lösen, beginnend mit Computational Fluid Dynamics (CFD) und Materials Deformation (MD), wobei weitere Anwendungsfälle in Kürze folgen.

Um die Differentialgleichungen anzugehen, werden die Testlösungen als Linearkombinationen orthogonaler Funktionen (typischerweise Chebyshev-Polynome, und genauer $2^n$ davon, wobei $n$ die Anzahl der Qubits ist, die deine Funktion kodieren) kodiert, parametrisiert durch die Winkel einer Variable Quantum Circuit (VQC). Der Ansatz erzeugt einen Zustand, der die Funktion kodiert, die durch Observablen ausgewertet wird, deren Kombinationen die Auswertung der Funktion an allen Punkten ermöglichen. Du kannst dann die Verlustfunktion auswerten, in der die Differentialgleichungen kodiert sind, und die Winkel in einer hybriden Schleife feinabstimmen, wie im Folgenden gezeigt. Die Testlösungen kommen den tatsächlichen Lösungen schrittweise näher, bis du ein zufriedenstellendes Ergebnis erreichst.

![Workflow der QUICK-PDE-Funktion](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Zusätzlich zu dieser hybriden Schleife kannst du auch verschiedene Optimierer verketten. Dies ist nützlich, wenn du einen globalen Optimierer verwenden möchtest, um einen guten Satz von Winkeln zu finden, und dann einen feineren Optimierer, um einem Gradienten zum besten Satz benachbarter Winkel zu folgen. Im Fall der Computational Fluid Dynamics (CFD) liefert die Standard-Optimierungssequenz die besten Ergebnisse - aber im Fall der Material Deformation (MD) kannst du, während der Standard gute Ergebnisse liefert, ihn für problemspezifische Vorteile weiter konfigurieren.

Beachte, dass wir für jede Variable der Funktion die Anzahl der Qubits angeben (mit der du experimentieren kannst). Durch das Stapeln von 10 identischen Schaltungen und die Auswertung der 10 identischen Observablen auf verschiedenen Qubits durch eine große Schaltung kannst du innerhalb des CMA-Optimierungsprozesses Rauschminderung durchführen, wobei du auf die Noise-Learner-Methode vertraust und die Anzahl der benötigten Shots erheblich reduzierst.
### Computational fluid dynamics

Die inviszide Burgers-Gleichung modelliert strömende nicht-viskose Flüssigkeiten wie folgt:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ stellt das Fluidgeschwindigkeitsfeld dar. Dieser Anwendungsfall hat eine zeitliche Randbedingung: Du kannst die Anfangsbedingung auswählen und dann das System relaxieren lassen. Derzeit sind die einzigen akzeptierten Anfangsbedingungen lineare Funktionen: $ax + b$.

Die Argumente für CFDs Differentialgleichungen befinden sich auf einem festen Gitter wie folgt:

- $t$ liegt zwischen 0 und 0,95 mit 30 Abtastpunkten. $x$ liegt zwischen 0 und 0,95 mit einer Schrittweite von 0,2375.

### Material deformation

Dieser Anwendungsfall konzentriert sich auf hypoelastische Verformung mit dem eindimensionalen Zugversuch, bei dem ein im Raum fixierter Stab an seinem anderen Ende gezogen wird. Wir beschreiben das Problem wie folgt:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ stellt den Kompressionsmodul des gedehnten Materials dar, $n$ den Exponenten eines Potenzgesetzes, $b$ die Kraft pro Masseneinheit, $\epsilon_0$ die Proportionalitätsspannungsgrenze, $\sigma_0$ die Proportionalitätsdehnungsgrenze, $u$ die Spannungsfunktion und $\sigma$ die Dehnungsfunktion.

Der betrachtete Stab hat eine Einheitslänge. Dieser Anwendungsfall hat eine Randbedingung für Oberflächenspannung $t$, oder die Menge an Arbeit, die benötigt wird, um den Stab zu dehnen.

Die Argumente für MDs Differentialgleichungen befinden sich auf einem festen Gitter wie folgt:

- $x$ liegt zwischen 0 und 1 mit einer Schrittweite von 0,04.

## Benchmarks

Die folgende Tabelle zeigt Statistiken zu verschiedenen Ausführungen unserer Funktion.

| Beispiel                              | Anzahl der Qubits | Initialisierung       | Fehler    | Gesamtzeit (min) | Runtime-Nutzung (min) |
| ------------------------------------- | ----------------- | --------------------- | --------- | ---------------- | --------------------- |
| Inviszide Burgers-Gleichung           | 50                | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66               | 25                    |
| Hypoelastischer 1D-Zugversuch         | 18                | `RANDOM`              | $10^{-2}$ | 123              | 100                   |
## Erste Schritte
Fülle das [Formular aus, um Zugang zur QUICK-PDE-Funktion anzufordern](https://forms.cloud.microsoft/e/3Wi9cbjQPK). Anschließend, vorausgesetzt du hast bereits [dein Konto gespeichert](/guides/functions#install-qiskit-functions-catalog-client) in deiner lokalen Umgebung, wähle die Funktion wie folgt aus:

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_cloud / ibm_quantum_platform",
    instance="USER_CRN / HGP",
    token="USER_API_KEY / IQP_API_TOKEN",
)

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

In [ ]:
quick = catalog.load("colibritd/quick-pde")

Überprüfe den [Status](/guides/functions#check-job-status) deiner Qiskit Function-Workload oder gib [Ergebnisse](/guides/functions#retrieve-results) wie folgt zurück:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(
    use_case="CFD_BURGER", physical_parameters={"a": 1.0, "b": 0.0}
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [ ]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())
solution = job.result()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_result_3d(result):
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")

    t, x = np.meshgrid(result["samples"]["t"], result["samples"]["x"])

    ax.plot_surface(
        t,
        x,
        result["functions"]["u"],
        edgecolor="royalblue",
        lw=0.25,
        rstride=26,
        cstride=26,
        alpha=0.3,
    )
    ax.scatter(t, x, result["functions"]["u"], marker=".")
    ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

    plt.show()


# Call
plot_result_3d(solution)

![Output of the previous code cell](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Material deformation

Der Material Deformation-Anwendungsfall erfordert die physikalischen Parameter deines Materials und die angewandte Kraft wie folgt:

In [ ]:
# Launches the solving for an arbitrary mu
job = quick.run(use_case="CFD_EULER", physical_parameters={"mu": 0.1})

solution = job.result()


# Colorplot function
def plot_result_2d(result):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    configs = {
        "g": {"cmap": "viridis", "title": "g(t, x)"},
        "u": {"cmap": "plasma", "title": "u(t, x)"},
    }

    t = result["samples"]["t"]
    x = result["samples"]["x"]

    for ax, (field, cfg) in zip(axes, configs.items()):
        v = result["functions"][field]

        im = ax.contourf(t, x, v, levels=50, cmap=cfg["cmap"])
        fig.colorbar(im, ax=ax, label=cfg["title"])

        ax.set_xlabel("t")
        ax.set_ylabel("x")
        ax.set_title(cfg["title"], fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()


plot_result_2d(solution)

![Output of the previous code cell](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

Das Folgende ist ein Beispiel dafür, wie du den Wert der Funktion für einen bestimmten Satz von Koordinaten erhältst:

In [ ]:
# Select the properties of your material
job = quick.run(
    use_case="MD",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# Plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

## Fehlermeldungen abrufen
Wenn der Status deiner Workload `ERROR` ist, verwende `job.error_message()`, um die Fehlermeldung zum Debuggen abzurufen, wie folgt:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Fetch error messages

If your workload status is `ERROR`, use `job.error_message()` to fetch the error message to help debug, as follows:

In [ ]:
job = quick.run(use_case="MD", physical_params={})

print(job.error_message())


# A wrapper can also be used for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

## Support erhalten

Für Support kontaktiere qiskit-function-support@colibritd.com.

## Nächste Schritte

> **Tip:** - Fülle das Formular aus, um [Zugang zur QUICK-PDE-Funktion anzufordern](https://forms.cloud.microsoft/e/3Wi9cbjQPK).
> - Besuche die [API-Referenz](https://docs.quantum.ibm.com/api/functions/colibritd-pde) für diese Qiskit Function.
> - Versuche, eine strömende nicht-viskose Flüssigkeit mit QUICK-PDE im [Tutorial](/tutorials/colibritd-pde) zu modellieren.
> - Lies [Jaffali, H., et al. (2025).  H-DES: a Quantum-Classical Hybrid Differential Equation Solver. arXiv preprint arXiv:2410.01130](https://arxiv.org/abs/2410.01130).